In [6]:
#LIBRERIAS
%pip install ib_async

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# CACAO (CC) - ACTUALIZACIÓN DE PRECIOS DESDE INTERACTIVE BROKERS
from ib_async import IB, Future
from pathlib import Path
from datetime import datetime, timedelta, date
import pandas as pd
import json, re, os, tempfile
import asyncio

# ========= CONFIG =========
HOST = "127.0.0.1"
PORT = 7496        # 7497 (paper) / 7496 (live)
CLIENT_ID = 17

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_cc.json")

# FECHA MÍNIMA - Solo traer datos desde esta fecha
START_DATE = "2025-01-01"

# Rango de años para generar contratos
START_YEAR = 2025  # Año inicial (contratos con datos históricos)
YEARS_AHEAD = 2    # Años hacia adelante desde el año actual
# ==========================

# Cacao (CC) meses: H=03 (Mar), K=05 (May), N=07 (Jul), U=09 (Sep), Z=12 (Dec)
MONTH_LETTERS = {'H': '03', 'K': '05', 'N': '07', 'U': '09', 'Z': '12'}
MONTH_NUM_TO_LETTER = {v: k for k, v in MONTH_LETTERS.items()}
ORDERED_MONTHS = ['H', 'K', 'N', 'U', 'Z']

# ---------- util contrato ----------
def cc_code_to_yyyymm(code: str) -> str:
    m = re.fullmatch(r"CC([HKNUZ])(\d{2})", code.strip().upper())
    if not m:
        raise ValueError(f"Código inválido CC: {code}")
    letter, yy = m.groups()
    yy_i = int(yy)
    year = 2000 + yy_i if yy_i <= 79 else 1900 + yy_i
    return f"{year}{MONTH_LETTERS[letter]}"

def yyyymm_to_cc_code(yyyymm: str) -> str:
    y = int(yyyymm[:4])
    mm = yyyymm[4:6]
    letter = MONTH_NUM_TO_LETTER[mm]
    return f"CC{letter}{str(y)[-2:]}"

def generate_cc_contracts() -> list:
    """Genera lista de contratos CC desde START_YEAR hasta años adelante"""
    contracts = []
    current_year = datetime.now().year
    end_year = current_year + YEARS_AHEAD
    
    for year in range(START_YEAR, end_year + 1):
        for letter in ORDERED_MONTHS:
            code = f"CC{letter}{str(year)[-2:]}"
            contracts.append(code)
    return contracts

async def qualify_cc_async(ib: IB, yyyymm: str):
    for ex in ("NYBOT", "ICEUS"):
        fut = Future(
            symbol="CC",
            lastTradeDateOrContractMonth=yyyymm,
            exchange=ex,
            currency="USD",
            includeExpired=True
        )
        try:
            qs = await ib.qualifyContractsAsync(fut)
            if qs:
                return qs[0]
        except Exception:
            pass
        await asyncio.sleep(0.05)
    return None

# ---------- históricos ----------
async def fetch_history_from_date(ib: IB, q, start_date: str):
    """Descarga histórico desde start_date hasta hoy"""
    start_dt = pd.to_datetime(start_date)
    end_dt = datetime.now()
    days = max(1, (end_dt - start_dt).days + 1)
    
    # Limitar a máximo 365 días para evitar timeouts
    days = min(days, 365)
    
    for what in ("TRADES", "MIDPOINT", "BID_ASK"):
        try:
            bars = await ib.reqHistoricalDataAsync(
                q, endDateTime="", durationStr=f"{days} D",
                barSizeSetting="1 day", whatToShow=what,
                useRTH=True, formatDate=2,  # useRTH=True para mercado regular
                timeout=30  # timeout más corto
            )
            if bars:
                rows = []
                for b in bars:
                    d = str(b.date)
                    if d >= start_date:
                        rows.append({
                            "date": d, "open": b.open, "high": b.high, "low": b.low,
                            "close": b.close, "volume": b.volume, "openinterest": None
                        })
                if rows:
                    df = pd.DataFrame(rows).drop_duplicates("date").sort_values("date")
                    return df.to_dict(orient="records")
        except Exception as e:
            pass
        await asyncio.sleep(0.3)
    return []

# ---------- JSON I/O ----------
def load_db(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except:
        return {}

def atomic_save(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name, dir=str(path.parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    except Exception:
        try:
            os.remove(tmp)
        finally:
            raise

# ---------- main async ----------
async def main_async():
    db = load_db(JSON_PATH)
    
    ib = IB()
    try:
        await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
    except Exception as e:
        print("No se pudo conectar a IBKR:", e)
        return

    try:
        today = datetime.now()
        
        # Contratos existentes en JSON
        cc_contracts = [k for k in db.keys() if re.fullmatch(r"CC[HKNUZ]\d{2}", k)]
        
        # Si no hay contratos, generar automáticamente
        if not cc_contracts:
            print("⚠️ JSON vacío - Generando contratos automáticamente...")
            cc_contracts = generate_cc_contracts()
            print(f"   Contratos generados: {len(cc_contracts)} (desde {START_YEAR})\n")
        
        print(f"Contratos a procesar: {len(cc_contracts)}")
        print(f"Fecha mínima: {START_DATE}\n")
        
        updated = 0
        skipped = 0
        
        for code in sorted(cc_contracts, key=lambda x: cc_code_to_yyyymm(x)):
            try:
                yyyymm = cc_code_to_yyyymm(code)
            except ValueError:
                continue
            
            q = await qualify_cc_async(ib, yyyymm)
            if not q:
                print(f"   {code}: no disponible en IBKR")
                skipped += 1
                continue
            
            # Obtener última fecha en JSON
            bars = db.get(code, [])
            bars = [r for r in bars if r["date"] >= START_DATE]
            last_date = max((r["date"] for r in bars), default=None)
            
            if last_date:
                start = (pd.to_datetime(last_date) + timedelta(days=1)).strftime("%Y-%m-%d")
                if start <= today.strftime("%Y-%m-%d"):
                    new_rows = await fetch_history_from_date(ib, q, start)
                    if new_rows:
                        merged = (pd.concat([pd.DataFrame(bars), pd.DataFrame(new_rows)], ignore_index=True)
                                  .drop_duplicates("date").sort_values("date"))
                        db[code] = merged.to_dict(orient="records")
                        print(f"✅ {code}: +{len(new_rows)} filas (hasta {merged.iloc[-1]['date']})")
                        updated += 1
                    else:
                        print(f"   {code}: al día ({last_date})")
                else:
                    print(f"   {code}: al día ({last_date})")
            else:
                rows = await fetch_history_from_date(ib, q, START_DATE)
                if rows:
                    db[code] = rows
                    print(f"✅ {code}: {len(rows)} filas nuevas")
                    updated += 1
                else:
                    print(f"⚠️ {code}: sin datos históricos")
            
            await asyncio.sleep(0.5)
        
        atomic_save(JSON_PATH, db)
        
        cc_with_data = [k for k in db.keys() if re.fullmatch(r"CC[HKNUZ]\d{2}", k) and db[k]]
        
        print(f"\n{'='*50}")
        print(f"Actualizados: {updated} | Con datos: {len(cc_with_data)} | Omitidos: {skipped}")
        print(f"Archivo: {JSON_PATH.name}")
        print(f"{'='*50}")
        
        if cc_with_data:
            print("\nÚltima fecha por contrato:")
            for code in sorted(cc_with_data, key=lambda x: cc_code_to_yyyymm(x)):
                bars = db.get(code, [])
                last = max((r["date"] for r in bars), default="N/A")
                print(f"  {code}: {last}")

    finally:
        if ib.isConnected():
            ib.disconnect()

await main_async()

Contratos a procesar: 15
Fecha mínima: 2025-01-01

   CCH25: al día (2025-03-17)
   CCK25: al día (2025-05-15)
   CCN25: al día (2025-07-17)
   CCU25: al día (2025-09-16)
   CCZ25: al día (2025-12-16)
✅ CCH26: +10 filas (hasta 2026-03-03)
✅ CCK26: +10 filas (hasta 2026-03-03)
✅ CCN26: +10 filas (hasta 2026-03-03)
✅ CCU26: +10 filas (hasta 2026-03-03)
✅ CCZ26: +10 filas (hasta 2026-03-03)
✅ CCH27: +10 filas (hasta 2026-03-03)
✅ CCK27: +10 filas (hasta 2026-03-03)
✅ CCN27: +10 filas (hasta 2026-03-03)
✅ CCU27: +10 filas (hasta 2026-03-03)
✅ CCZ27: +9 filas (hasta 2026-03-02)

Actualizados: 10 | Con datos: 15 | Omitidos: 0
Archivo: data_cc.json

Última fecha por contrato:
  CCH25: 2025-03-17
  CCK25: 2025-05-15
  CCN25: 2025-07-17
  CCU25: 2025-09-16
  CCZ25: 2025-12-16
  CCH26: 2026-03-03
  CCK26: 2026-03-03
  CCN26: 2026-03-03
  CCU26: 2026-03-03
  CCZ26: 2026-03-03
  CCH27: 2026-03-03
  CCK27: 2026-03-03
  CCN27: 2026-03-03
  CCU27: 2026-03-03
  CCZ27: 2026-03-02


In [3]:
# PRECIOS DEL ÚLTIMO DÍA DEL MES ANTERIOR - TODOS LOS FUTUROS DE CACAO
from datetime import date
from dateutil.relativedelta import relativedelta
import pandas as pd
import json
from pathlib import Path
import re

JSON_PATH = Path(r"C:\Users\DanielAristizabal\Saman\Banca de Inversión - Documents\Super de Alimentos\Gestión de riesgo\Data\data_cc.json")

# Cargar datos
with open(JSON_PATH, "r", encoding="utf-8") as f:
    db = json.load(f)

# Calcular mes anterior
hoy = date.today()
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - relativedelta(days=1)
mes_anterior = ultimo_dia_mes_anterior.strftime("%Y-%m")

print(f"Buscando precios del mes: {mes_anterior}")
print(f"(Último día del mes anterior: {ultimo_dia_mes_anterior})\n")

# Recopilar precios del último día del mes anterior para cada contrato
resultados = []

cc_contracts = [k for k in db.keys() if re.fullmatch(r"CC[HKNUZ]\d{2}", k)]

for code in sorted(cc_contracts):
    bars = db.get(code, [])
    
    # Filtrar solo fechas del mes anterior
    fechas_mes = [r for r in bars if r["date"].startswith(mes_anterior)]
    
    if fechas_mes:
        # Obtener el último día disponible del mes
        ultimo_registro = max(fechas_mes, key=lambda x: x["date"])
        resultados.append({
            "Contrato": code,
            "Fecha": ultimo_registro["date"],
            "Close": ultimo_registro["close"],
            "Volume": ultimo_registro["volume"]
        })
    else:
        resultados.append({
            "Contrato": code,
            "Fecha": "Sin datos",
            "Close": None,
            "Volume": None
        })

# Crear DataFrame
df_ultimo_dia_mes = pd.DataFrame(resultados)
print(f"{'='*60}")
print(f"PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR ({mes_anterior})")
print(f"{'='*60}")
display(df_ultimo_dia_mes)

Buscando precios del mes: 2026-02
(Último día del mes anterior: 2026-02-28)

PRECIOS ÚLTIMO DÍA DEL MES ANTERIOR (2026-02)


,Contrato,Fecha,Close,Volume
0,CCH25,Sin datos,NaN,NaN
1,CCH26,2026-02-27,2798.0,0.0
2,CCH27,2026-02-27,3171.0,1946.0
3,CCK25,Sin datos,NaN,NaN
4,CCK26,2026-02-27,2888.0,14184.0
5,CCK27,2026-02-27,3225.0,504.0
6,CCN25,Sin datos,NaN,NaN
7,CCN26,2026-02-27,2942.0,6232.0
8,CCN27,2026-02-27,3260.0,145.0
9,CCU25,Sin datos,NaN,NaN
